In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression 
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor , StackingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub

In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="food-delivery-time-prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/food-delivery-time-prediction"

Repository mridul0010/food-delivery-time-prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow")

In [4]:
mlflow.set_experiment("5. Stacking Regressor Hyperparameter Tuning")

2026/07/05 22:08:57 INFO mlflow.tracking.fluent: Experiment with name '5. Stacking Regressor Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/111227dca0484483a7fcb1fa3e5c03d3', creation_time=1783269538785, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1783269538785, lifecycle_stage='active', name='5. Stacking Regressor Hyperparameter Tuning', tags={}, trace_location=None, workspace='default'>

In [5]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [15]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [16]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [17]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [18]:
pt = PowerTransformer()

In [19]:
best_xgb_params = {
    'n_estimators': 245,
    'max_depth': 23,
    'learning_rate': 0.19864909172366368,
    'subsample': 0.9180765181845632,
    'min_child_weight': 7,
    'gamma': 1.0407049003047555,
    'reg_lambda': 22.40266224283313
}

best_rf_params = {
    'n_estimators': 299,
    'max_depth': 13,
    'min_samples_split': 20,
    'min_samples_leaf': 2,
    'max_features': 0.44603640869435546,
    'bootstrap': False
}

best_rf = RandomForestRegressor(**best_rf_params)
best_xgb = XGBRegressor(**best_xgb_params)

In [20]:
def objective(trial):
    with mlflow.start_run(nested=True):
        meta_model_name = trial.suggest_categorical("model",["LR","KNN","DT"])

        if meta_model_name == "LR":
            meta = LinearRegression()

        elif meta_model_name == "KNN":
            n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,15)
            weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
            meta = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                        weights=weights_knn,n_jobs=-1)

        elif meta_model_name == "DT":
            max_depth_dt = trial.suggest_int("max_depth_dt",1,10)
            min_samples_split_dt = trial.suggest_int("min_samples_split_dt",2,10)
            min_samples_leaf_dt = trial.suggest_int("min_samples_leaf_dt",1,10)
            meta = DecisionTreeRegressor(max_depth=max_depth_dt,
                                        min_samples_split=min_samples_split_dt,
                                        min_samples_leaf=min_samples_leaf_dt,
                                        random_state=42)

        # log meta model name
        mlflow.log_param("meta_model_name",meta_model_name)

        # stacking regressor
        stacking_reg = StackingRegressor(estimators=[("rf",best_rf),
                                                    ("xgb",best_xgb)],
                                        final_estimator=meta,
                                        cv=5,n_jobs=-1)

        # build transformed regressor
        model = TransformedTargetRegressor(regressor=stacking_reg,
                                            transformer=pt)

        # train the model
        model.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_test = model.predict(X_test_trans)

        # mean absoulte error
        error = mean_absolute_error(y_test,y_pred_test)

        # log error
        mlflow.log_metric("MAE",error)

        return error

In [21]:
# create optuna study
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=20,n_jobs=1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

[I 2026-07-05 22:08:58,492] A new study created in memory with name: no-name-f7769b0b-063f-495e-b963-a11cf532bacd
  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run burly-dolphin-586 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5/runs/6083342a0a5944f18afefa5417d9dcf7
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5


Best trial: 0. Best value: 3.52372:   5%|▌         | 1/20 [01:25<27:06, 85.59s/it]

[I 2026-07-05 22:10:25,300] Trial 0 finished with value: 3.523720458664059 and parameters: {'model': 'DT', 'max_depth_dt': 2, 'min_samples_split_dt': 3, 'min_samples_leaf_dt': 8}. Best is trial 0 with value: 3.523720458664059.
🏃 View run lyrical-elk-79 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5/runs/73276c98f1094993a630c992bebf2414
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5


Best trial: 1. Best value: 2.98048:  10%|█         | 2/20 [03:35<33:24, 111.38s/it]

...

[I 2026-07-05 22:48:10,789] Trial 17 finished with value: 2.9809359606913235 and parameters: {'model': 'LR'}. Best is trial 1 with value: 2.9804820539825703.
🏃 View run fortunate-mink-890 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5/runs/7c81e8cd96644350b242679efc6c1309
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5


Best trial: 18. Best value: 2.97958:  95%|█████████▌| 19/20 [40:18<01:44, 104.21s/it]

[I 2026-07-05 22:49:18,052] Trial 18 finished with value: 2.9795811097499776 and parameters: {'model': 'LR'}. Best is trial 18 with value: 2.9795811097499776.
🏃 View run victorious-conch-246 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5/runs/e7b36cf8ff8e44e387b8b1fb12ee0bc1
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5


Best trial: 18. Best value: 2.97958: 100%|██████████| 20/20 [41:23<00:00, 124.17s/it]


[I 2026-07-05 22:50:23,190] Trial 19 finished with value: 3.1026733680647154 and parameters: {'model': 'KNN', 'n_neighbors_knn': 14, 'weights_knn': 'distance'}. Best is trial 18 with value: 2.9795811097499776.
🏃 View run best_model at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5/runs/5d7f514bd4b943159a2dfbd09fa89654
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/5


In [22]:
study.best_value

2.9795811097499776

In [23]:
study.best_params

{'model': 'LR'}

In [24]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_max_depth_dt,params_min_samples_leaf_dt,params_min_samples_split_dt,params_model,params_n_neighbors_knn,params_weights_knn,state
0,0,3.523720,2026-07-05 22:08:59.716722,2026-07-05 22:10:25.300105,0 days 00:01:25.583383,2.0,8.0,3.0,DT,NaN,NaN,COMPLETE
1,1,2.980482,2026-07-05 22:10:25.321586,2026-07-05 22:12:34.734562,0 days 00:02:09.412976,NaN,NaN,NaN,LR,NaN,NaN,COMPLETE
2,2,4.075798,2026-07-05 22:12:34.746033,2026-07-05 22:15:10.314829,0 days 00:02:35.568796,NaN,NaN,NaN,KNN,1.0,uniform,COMPLETE
3,3,2.981496,2026-07-05 22:15:10.317918,2026-07-05 22:17:23.600966,0 days 00:02:13.283048,NaN,NaN,NaN,LR,NaN,NaN,COMPLETE
4,4,3.256771,2026-07-05 22:17:23.603354,2026-07-05 22:18:28.826573,0 days 00:01:05.223219,NaN,NaN,NaN,KNN,5.0,distance,COMPLETE


In [25]:
study.trials_dataframe()['params_model'].value_counts()

params_model
LR     12
DT      4
KNN     4
Name: count, dtype: int64